# GPU Setup

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("device:", device)

if torch.cuda.is_available():
    print("GPU: ",torch.cuda.get_device_name(0))

torch.manual_seed(42)

device: cuda
GPU:  Tesla T4


# Sample Restaurant Corpus 1

In [40]:
sentences = [
    "i want chinese food",
    "i want korean food",
    "i want italian food",
    "i want japanese food",
    "i want thai food",
    "i want indian food",
    "i want a chinese restaurant",
    "i want a korean restaurant",
    "i want an italian restaurant",
    "i want a japanese restaurant",
    "i want to eat chinese food",
    "i want to eat korean food",
    "i want to eat italian food",
    "i want to eat japanese food",
    "i would like chinese food",
    "i would like korean food",
    "i would like italian food",
    "i would like japanese food",
    "find a chinese restaurant",
    "find a korean restaurant",
    "find an italian restaurant",
    "find a japanese restaurant",
    "show me a chinese restaurant",
    "show me a korean restaurant",
    "show me an italian restaurant",
    "show me a japanese restaurant",
    "can you find a chinese restaurant",
    "can you find a korean restaurant",
    "can you find an italian restaurant",
    "can you find a japanese restaurant",
    "where can i eat chinese food",
    "where can i eat korean food",
    "where can i eat italian food",
    "where can i eat japanese food",
    "i want cheap chinese food",
    "i want cheap korean food",
    "i want expensive italian food",
    "find a cheap restaurant",
    "find a good restaurant",
    "find a nearby restaurant",
    "i want a restaurant downtown",
    "i want a restaurant nearby",
    "show me a restaurant downtown",
    "the chinese restaurant is good",
    "the korean restaurant is good",
    "the italian restaurant is expensive",
    "the thai restaurant is cheap",
]

# Build the Voca


In [41]:
# pad : 짧은 문장 보정, bos : 문장 시작, eos : 문장의 끝, unk : 모르는 단어
SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>"
]

words = []

for sentence in sentences:
    words.extend(
            sentence.lower().split()
    )

vocab = SPECIAL_TOKENS + sorted(
    set(words)
)

word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}


# 상수로 정의 해주고요
PAD_ID = word_to_id["<pad>"]
BOS_ID = word_to_id["<bos>"]
EOS_ID = word_to_id["<eos>"]
UNK_ID = word_to_id["<unk>"]

vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print(word_to_id)

Vocabulary size: 33
{'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3, 'a': 4, 'an': 5, 'can': 6, 'cheap': 7, 'chinese': 8, 'downtown': 9, 'eat': 10, 'expensive': 11, 'find': 12, 'food': 13, 'good': 14, 'i': 15, 'indian': 16, 'is': 17, 'italian': 18, 'japanese': 19, 'korean': 20, 'like': 21, 'me': 22, 'nearby': 23, 'restaurant': 24, 'show': 25, 'thai': 26, 'the': 27, 'to': 28, 'want': 29, 'where': 30, 'would': 31, 'you': 32}


# Prepare Next-word Training Data
다음 단어 예측용 학습 데이터 만들기

In [42]:
MAX_LEN = 12

class LanguageModelDataset(Dataset):
    def __init__(self, sentences):
        self.examples = []

        for sentence in sentences:
            # 토큰화
            tokens = sentence.lower().split()

            # ID로 변환하기
            ids = [BOS_ID] # 문장 시작 표시
            for word in tokens:
                # 모르는 거는 모른다고 번호 입력
                ids.append(word_to_id.get(word,UNK_ID))
            ids.append(EOS_ID) # 문장 종료 표시

            # 입력
            x = ids[:-1] # BOS_ID 부터 EOS_ID 전까지 EOS_ID는 버림
            # 다음 단어 타겟
            y = ids[1:] # x[t]를 보고 y[t](= 다음 단어)를 맞히도록 짝을 맞춤

            # 긴거는 자르고 짧은거는 패딩을 붙이고
            x = x[:MAX_LEN]
            y = y[:MAX_LEN]
            padding = MAX_LEN - len(x)
            x = x + [PAD_ID] * padding
            y = y + [PAD_ID] * padding

            # 텐서로 변환하기
            self.examples.append(
                (
                    torch.tensor( x, dtype=torch.long ),
                    torch.tensor( y, dtype=torch.long )
                )
            )

    # 샘플 개수 반환 함수
    def __len__(self):
        return len(self.examples)

    # idx번째 (x, y) 쌍 꺼내기
    def __getitem__(self, index):
        return self.examples[index]

dataset = LanguageModelDataset( sentences )

# loader의 역할
# 16개의 문장 배치로 묵기
# 문장 섞기
loader = DataLoader(
    dataset,
    batch_size = 16,
    shuffle=True
)

x, y = dataset[0]
print("Input IDs :", x)
print("Target IDs :", y)

Input IDs : tensor([ 1, 15, 29,  8, 13,  0,  0,  0,  0,  0,  0,  0])
Target IDs : tensor([15, 29,  8, 13,  2,  0,  0,  0,  0,  0,  0,  0])


# Model Training 1
모델 훈련

In [43]:
def train_model(
        model,
        loader,
        epochs=200,
        learning_rate=0.003
):

    model = model.to(device)

    # loss 구하기, PAD_ID는 무시
    criterion = nn.CrossEntropyLoss(
        ignore_index=PAD_ID
    )

    # optimizer 구하기 , AdamW 사용하기
    optimizer = torch.optim.AdamW(
        # 모델의 모든 파라미터를 대상으로 진행하겠다는 의미
        model.parameters(),
        lr=learning_rate
    )

    model.train()

## AdamW 공부

### Adam 이란? - (Adaptive Moment Estimation)

- Adam 옵티마이저는 딥러닝 모델 학습에 널리 사용되는 최적화 알고리즘
- Stochastic Gradient Descent(SGD, 확률적 경사하강법)의 확장 기법인 **Momentum**(1차 모멘트)과 **RMSProp**(2차 모멘트)의 장점을 결합
  - 기울기의 1차, 2차 모멘트 추정치로 **파라미터마다 학습률을 개별적으로 조절** → 기존 방식보다 빠르고 안정적으로 수렴
  


- **Momentum**: 이전 기울기 방향을 기억해서 관성을 줌 → 진동이 줄고 빨리 내려감
  - 비유: 언덕을 굴러가는 볼링공. 작은 요철에 흔들리지 않음

- **AdaGrad**: 지금까지 기울기 제곱을 **전부 누적**해서, 많이 움직인 파라미터는 보폭을 줄임
  - 문제: 누적만 하니까 G가 계속 커져서 **학습률이 0에 가까워지고 학습이 멈춤**

- **RMSProp**: AdaGrad의 "전부 누적"을 **최근 위주 이동평균**으로 바꿈 → 오래된 기울기는 서서히 잊음
  - 학습률이 0으로 죽지 않음

- **Adam**: Momentum의 m(방향) + RMSProp의 v(보폭)
  - $m = \beta_1 m + (1-\beta_1) g$ → **1차 모멘트** = 기울기의 이동평균 (어느 방향으로 갈지)
  - $v = \beta_2 v + (1-\beta_2) g^2$ → **2차 모멘트** = 기울기 제곱의 이동평균 (얼마나 흔들리는지)
  - $w = w - lr \cdot \frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$
  - $\hat{m}, \hat{v}$: 초반에 m, v가 0에서 시작해 작게 나오는 걸 보정한 값 (bias correction)
  - $\epsilon$: 분모가 0이 되는 것을 막는 아주 작은 수 (보통 $10^{-8}$)
  - 기본값: $lr = 0.001$, $\beta_1 = 0.9$, $\beta_2 = 0.999$


---
비유로 알아보자

| 옵티마이저 | 등산객 |
|---|---|
| SGD | 발밑 경사만 보고 한 걸음씩 |
| Momentum | 내려오던 관성을 살려서 성큼성큼 |
| AdaGrad | 걸어온 총 거리만큼 지쳐서 보폭이 줄다가 결국 멈춤 |
| RMSProp | 최근 지형만 보고 보폭 조절 → 지치지 않음 |
| Adam | 관성 + 최근 지형 보고 보폭 조절 |
| AdamW | Adam + 배낭 무게(가중치)를 매번 일정 비율로 덜어냄 |

---
### AdamW 란? - (Adam with decoupled Weight decay)

- Adam에 **Weight Decay를 "분리(decoupling)"해서** 적용한 옵티마이저
- 현재 LLM 학습의 표준 (GPT, Llama 등)

#### 먼저, Weight Decay란?
- 가중치가 너무 커지지 않게 **매 스텝 조금씩 0 쪽으로 줄이는 것**
- 목적: 가중치가 커지면 학습 데이터를 외워 버림(과적합) -> 이를 방지
  - $w = w - lr \cdot \lambda \cdot w$ ($\lambda$ = weight decay 계수)
  - 비유: 화분 가지치기. 가지(가중치)가 제멋대로 뻗지 않게 매번 다듬기

#### 기존 Adam의 문제 (L2 Regularization 방식)
- Weight decay를 **손실 함수에 직접 더해서** 구현함
  - 손실 = 원래 손실 + $\frac{\lambda}{2} w^2$ → 기울기 = $g + \lambda w$
- 그런데 Adam은 기울기를 $\sqrt{v}$로 나눔 → **decay 항까지 같이 나눠져 버림**
  - 과거 기울기가 컸던 파라미터 ($\sqrt{v}$ 큼) → decay가 **약해짐**
  - 과거 기울기가 작았던 파라미터 ($\sqrt{v}$ 작음) → decay가 **세짐**
- 결과: 같은 $\lambda$를 줬는데 **파라미터마다 규제 강도가 제멋대로 달라짐**

#### AdamW의 해결
- Weight decay를 기울기 계산에서 **분리**해서 **맨 마지막에 따로** 적용
  1. Adam 업데이트는 **순수한 손실 기울기만으로** 계산
     - $w = w - lr \cdot \frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$
  2. 그다음 weight decay를 **가중치에서 직접 빼기** ($\sqrt{v}$로 안 나눔!)
     - $w = w - lr \cdot \lambda \cdot w$
- 결과: 모든 파라미터가 **자기 크기에 비례해서 같은 비율로** 줄어듦

#### 비유: 월급과 세금
- Adam + L2: 세금을 소득 계산에 섞은 뒤 사람마다 다른 보정률($\sqrt{v}$)로 나눔 → 고소득자일수록 세금이 깎이는 불공평한 세제
- AdamW: 소득 보정은 보정대로, 세금은 **모든 재산에 같은 세율로 따로** 걷음 → 공평

#### 코드
```python
optimizer = torch.optim.AdamW(
    model.parameters(),    # 모델의 모든 파라미터(W, b)를 대상으로
    lr=learning_rate,      # 학습률 (보폭)
    weight_decay=0.01      # 생략 시 기본값 0.01 (Adam은 기본값 0)
)
```

- 실무: LLM은 보통 `weight_decay=0.1`, `betas=(0.9, 0.95)` 사용